# Multi-3x High-Chop Risk Control V1

## tl;dr

本卡把 16 只长期股票类 3x ETF 的高位震荡拆成 benign、breakout、breakdown、whipsaw 四类路径状态，再以验证集选择的模型驱动 SOXL/SOXX/现金控制器。首轮样本外结果显著降低波动与回撤，但上行捕获、风险仓位和跨 underlying 迁移门槛未过，因此不晋升。

## Context & Methods

### Key Assumptions

- 数据截止 2026-07-02，信号使用收盘时已知数据，目标从下一交易日开盘生效。
- 数据通过 `qlib.init` 与 `qlib.data.D.features` 回读；模型分段使用 `DatasetH`、`DataHandlerLP`、`QlibDataLoader`。
- 2023–2026 为外层 SOXL 样本外年份；内层使用滚动三年、leave-one-underlying-out 验证，每类训练/验证样本下限为 50/20。
- 只允许 SOXL、SOXX、现金，单边换手成本基准为 3bp；本研究不包含真实下单。

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if ROOT.as_posix() not in sys.path:
    sys.path.insert(0, ROOT.as_posix())

from src.workflow.multi_3x_high_chop_risk_control import (
    EXPERIMENT,
    RESEARCH_ETFS,
    run_multi_3x_high_chop_risk_control_card,
)

CARD = ROOT / 'research' / EXPERIMENT
AS_OF = '2026-07-02'
print({'experiment': EXPERIMENT, 'as_of': AS_OF, 'research_etf_count': len(RESEARCH_ETFS)})

{'experiment': 'multi_3x_high_chop_risk_control_v1', 'as_of': '2026-07-02', 'research_etf_count': 16}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


## Data

下面执行完整 qlib provider 回读、DatasetH 分段、验证集校准与选模、外层 SOXL 预测、控制器回测及 Recorder 记录。

In [2]:
summary = run_multi_3x_high_chop_risk_control_card(root=ROOT)
assert summary['qlib_contract']['data_api'] == ['qlib.init', 'qlib.data.D.features']
assert summary['qlib_contract']['dataset_api'] == ['DatasetH', 'DataHandlerLP', 'QlibDataLoader']
assert summary['model_selection']['selection_uses_outer_test'] is False
print({'status': summary['status'], 'recorder_id': summary['recorder']['recorder_id']})

[70818:MainThread](2026-07-19 15:05:07,971) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[70818:MainThread](2026-07-19 15:05:08,444) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[70818:MainThread](2026-07-19 15:05:08,445) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/multi-3x-high-chop-risk-control-v1/data/qlib/soxl')}


[70818:MainThread](2026-07-19 15:05:08,468) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[70818:MainThread](2026-07-19 15:05:08,470) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


...

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[70818:MainThread](2026-07-19 15:05:46,592) INFO - qlib.workflow - [exp.py:258] - Experiment 598612766348301076 starts running ...


[70818:MainThread](2026-07-19 15:05:46,675) INFO - qlib.workflow - [recorder.py:345] - Recorder 4e4506ceaca044b79d546723ea90dec9 starts running under Experiment 598612766348301076 ...


[70818:MainThread](2026-07-19 15:05:46,890) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


{'status': 'completed', 'recorder_id': '4e4506ceaca044b79d546723ea90dec9'}


## Results

In [3]:
selected = json.loads((CARD / 'selected_model.json').read_text(encoding='utf-8'))
benchmarks = pd.read_csv(CARD / 'backtest_benchmark_comparison.csv')
risk = pd.read_csv(CARD / 'risk_analysis.csv')
display(pd.DataFrame(selected['outer_models'])[['outer_year', 'candidate', 'validation_log_loss']])
display(benchmarks[['benchmark', 'annualized_return', 'annualized_volatility', 'max_drawdown', 'return_to_drawdown']])
display(risk.loc[risk['metric'].str.startswith('gate_')])

,outer_year,candidate,validation_log_loss
0,2023,lightgbm_multiclass,1.188250
1,2024,lightgbm_multiclass,1.195542
2,2025,lightgbm_multiclass,1.181616
3,2026,lightgbm_multiclass,1.135319


,benchmark,annualized_return,annualized_volatility,max_drawdown,return_to_drawdown
0,soxl_buy_hold,1.416331,1.073270,-0.887445,1.595965
1,soxx_buy_hold,0.596637,0.359905,-0.430919,1.384568
2,constant_soxl_average_exposure,0.090120,0.064803,-0.097856,0.920938
3,matched_beta_static,0.134940,0.093504,-0.135486,0.995964
4,base_constrained_ref,0.228426,0.219718,-0.293414,0.778512
5,selected_high_chop_base_throttle,0.178737,0.162048,-0.227776,0.784703
6,rule_only_risk_controller,0.090946,0.106973,-0.111129,0.818380


,metric,value,status
12,gate_drawdown_absolute,-0.095897,pass
13,gate_drawdown_relative_improvement,0.673168,pass
14,gate_annualized_volatility,0.115908,pass
15,gate_cvar_95,-0.017561,pass
16,gate_worst_five_session_return,-0.076250,pass
17,gate_upside_capture,0.087623,fail
18,gate_average_risky_weight,0.140247,fail
19,gate_return_to_drawdown_benchmarks,NaN,fail
20,gate_annual_fold_gate,4.000000,pass
21,gate_stress_10bps,NaN,pass


## Takeaways

In [4]:
metrics = summary['stitched_metrics']
failed = [name for name, gate in summary['promotion']['gates'].items() if not gate['passed']]
print(
    f"执行结论：年化收益 {metrics['annualized_return']:.2%}，年化波动 {metrics['annualized_volatility']:.2%}，"
    f"最大回撤 {metrics['max_drawdown']:.2%}，5% CVaR {metrics['cvar_95']:.2%}。"
)
print(f"晋升决定：{summary['promotion_decision']}；失败门槛：{', '.join(failed)}。")
print('解释：风险控制有效，但上行捕获和跨资产 breakdown 迁移能力不足，当前模型更像低风险现金化器，而不是合格的高位震荡控制层。')
print('下一步：提高正常/突破状态的基础风险预算，重做 breakdown 概率阈值校准，并保留严格样本外门槛；不进入真实交易。')

执行结论：年化收益 12.07%，年化波动 11.59%，最大回撤 -9.59%，5% CVaR -1.76%。
晋升决定：not_promoted_risk_control_gates_failed；失败门槛：upside_capture, average_risky_weight, return_to_drawdown_benchmarks, transfer_fold_pass_rate。
解释：风险控制有效，但上行捕获和跨资产 breakdown 迁移能力不足，当前模型更像低风险现金化器，而不是合格的高位震荡控制层。
下一步：提高正常/突破状态的基础风险预算，重做 breakdown 概率阈值校准，并保留严格样本外门槛；不进入真实交易。
